# DiffusionGemma GGUF — Colab T4
Runs DiffusionGemma via `llama.cpp` (GGUF, Q4_K_M ~16.8GB) instead of the `transformers`/Triton W4A16 path, which hard-OOMs and can't multi-GPU-dispatch on a T4.

**Why this might work where the others didn't:**
- `llama.cpp` supports partial CPU+GPU layer offload (`-ngl`) — it spills overflow into system RAM instead of crashing, unlike the custom `trust_remote_code` Triton loader.
- Q4_K_M is 16.8GB — slightly over the T4's 15GB VRAM alone, but should fit once split across VRAM (15GB) + Colab's system RAM (~12.7GB).

**Known risks — read before running:**
- This uses **llama.cpp PR #24423** (unmerged, DiffusionGemma support). It's not mainline — build failures or CLI flag changes are possible since this is actively being developed upstream.
- `llama-diffusion-cli` is a new, diffusion-specific runner (not the standard `llama-cli`). Its exact non-interactive/batch behavior is not fully documented — the smoke test cell verifies it actually works before you trust the batch-eval cell.
- Build takes several minutes (CUDA compile). GGUF download is ~16.8GB.
- Requires `HF_TOKEN` with Gemma license accepted on both `google/diffusiongemma-26B-A4B-it` and `unsloth/diffusiongemma-26B-A4B-it-GGUF`.
- Accelerator: **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# Cell 1 — GPU + disk check
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!df -h /content | tail -1

In [ ]:
# Cell 2 — HF auth (Colab Secrets, falls back to manual prompt)
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Enter HF_TOKEN: ")

!pip install -q huggingface_hub sacrebleu

In [ ]:
# Cell 3 — Clone + build llama.cpp with DiffusionGemma support (PR #24423)
# Plain git fetch of the PR ref — avoids needing an authenticated `gh` CLI.
%cd /content
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
%cd /content/llama.cpp
!git fetch origin pull/24423/head:pr24423
!git checkout pr24423

!cmake -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release
!cmake --build build -j$(nproc) --config Release --target llama-diffusion-cli

!ls -la /content/llama.cpp/build/bin/llama-diffusion-cli

In [ ]:
# Cell 4 — Download the Q4_K_M GGUF (smallest available quant, ~16.8GB)
from huggingface_hub import hf_hub_download

MODEL_REPO = "unsloth/diffusiongemma-26B-A4B-it-GGUF"
GGUF_FILE = "diffusiongemma-26B-A4B-it-Q4_K_M.gguf"

gguf_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=GGUF_FILE,
    token=os.environ["HF_TOKEN"],
    local_dir="/content/models",
)
print(f"Downloaded to {gguf_path}")

In [ ]:
# Cell 5 — Smoke test. Verifies the binary actually loads + generates before
# trusting the batch-eval cell below.
#
# N_GPU_LAYERS: 99 = try to put everything on GPU (will likely partially fail
# since Q4_K_M is 16.8GB > 15GB VRAM — llama.cpp should fall back gracefully
# and spill remaining layers to system RAM rather than crash). If you see a
# hard CUDA OOM instead, LOWER this value (try 20, then 10, ...) until it loads.
N_GPU_LAYERS = 99

BIN = "/content/llama.cpp/build/bin/llama-diffusion-cli"

!echo "Translate the following English sentence to Russian. Output only the translation, nothing else.\nThe weather today is beautiful." | \
  {BIN} -m {gguf_path} -ngl {N_GPU_LAYERS} -cnv -n 150

In [ ]:
# Cell 6 — Batch eval on WMT14 EN->RU (3003 lines), only after Cell 5 confirms
# real translated output. Runs one subprocess call per line (model stays
# loaded between calls would be faster, but llama-diffusion-cli's batch/server
# mode isn't confirmed yet on this unmerged PR — start simple, verify output,
# then optimize if this is too slow).
import subprocess, time, sacrebleu
from pathlib import Path

data_dir = Path("/content/data/en-ru")
data_dir.mkdir(parents=True, exist_ok=True)
src_path = sacrebleu.get_source_file("wmt14", "en-ru")
ref_path = sacrebleu.get_reference_files("wmt14", "en-ru")[0]
import shutil
shutil.copy(src_path, data_dir / "test.en")
shutil.copy(ref_path, data_dir / "test.ru")

src_lines = (data_dir / "test.en").read_text().splitlines()
ref_lines = (data_dir / "test.ru").read_text().splitlines()

def translate(text, src_lang="English", tgt_lang="Russian"):
    prompt = (
        f"Translate the following {src_lang} sentence to {tgt_lang}. "
        f"Output only the {tgt_lang} translation, nothing else.\n{text}"
    )
    result = subprocess.run(
        [BIN, "-m", gguf_path, "-ngl", str(N_GPU_LAYERS), "-cnv", "-n", "150"],
        input=prompt, capture_output=True, text=True, timeout=300,
    )
    return result.stdout.strip()

# Timed smoke test first — 5 lines, to gauge per-line latency before committing
# to all 3003 (this model's generation speed on a T4 with partial CPU offload
# is unbenchmarked).
times = []
for src in src_lines[:5]:
    t0 = time.time()
    hyp = translate(src)
    dt = time.time() - t0
    times.append(dt)
    print(f"SRC: {src}\nHYP: {hyp}\n({dt:.1f}s)\n")

avg = sum(times) / len(times)
print(f"Avg {avg:.1f}s/line -> est. {avg*len(src_lines)/3600:.1f}h for all {len(src_lines)} lines")

In [ ]:
# Cell 7 — Full run + BLEU (run only after Cell 6's timing estimate looks acceptable)
from tqdm.notebook import tqdm

hyps = [translate(line) for line in tqdm(src_lines, desc="en-ru")]

out_dir = Path("/content/results/en-ru")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "hyps.ru").write_text("\n".join(hyps))

bleu = sacrebleu.corpus_bleu(hyps, [ref_lines], tokenize="13a")
report = f"SacreBLEU (13a): {bleu.score:.2f}\nModel: {MODEL_REPO}/{GGUF_FILE}\nLines: {len(hyps)}\n"
(out_dir / "bleu_report.txt").write_text(report)
print(report)